In [1]:
import pandas as pd
import os

# ==========================================
# 1. Configuration and Paths
# ==========================================
# Define the base project directory
BASE_PATH = '..' 

# Input and output file paths
# Note: Using 120m as provided in your latest snippet
INPUT_CSV = os.path.join(BASE_PATH, 'includes', 'dados','Tabela_consumo_Itapua_120m.csv')
OUTPUT_CSV = os.path.join(BASE_PATH, 'includes', 'Tabela_consumo_medio_Itapua_2025_12m.csv')

def main():
    print("--- Calculating Base Year Consumption Average (2025) ---")

    # ==========================================
    # 2. Data Loading and Filtering
    # ==========================================
    if not os.path.exists(INPUT_CSV):
        print(f"Error: Input file not found at {INPUT_CSV}")
        return

    # Load the CSV file (using semicolon as delimiter)
    df = pd.read_csv(INPUT_CSV, delimiter=';')

    # Convert AM_REFERENCIA to integer to facilitate filtering
    # Expected format: YYYYMM (e.g., 202501)
    df['AM_REFERENCIA'] = df['AM_REFERENCIA'].astype(int)

    # Filter records strictly within the year 2025
    df_2025 = df[(df['AM_REFERENCIA'] >= 202501) & (df['AM_REFERENCIA'] <= 202512)].copy()

    print(f"Total records found for the year 2025: {len(df_2025)}")

    if df_2025.empty:
        print("Warning: No records found for 2025. Please check the AM_REFERENCIA format.")
        return

    # ==========================================
    # 3. Processing the Average Consumption
    # ==========================================
    # Group by SK_MATRICULA (User ID) and calculate the mean of HCLQTCON (Consumption)
    # This generates the monthly average consumption used to initialize agents in Jan/2026
    #mean_consumption_2025 = df_2025.groupby('SK_MATRICULA')['HCLQTCON'].last().reset_index()
    # Calculate MEAN consumption per matriculation
    mean_consumption = df_2025.groupby('SK_MATRICULA')['HCLQTCON'].mean().reset_index()
    mean_consumption.columns = ['SK_MATRICULA', 'MEAN_CONSUMPTION']

    # Calculate LAST consumption per matriculation
    last_consumption = df_2025.groupby('SK_MATRICULA')['HCLQTCON'].last().reset_index()
    last_consumption = last_consumption[['SK_MATRICULA', 'HCLQTCON']].copy()
    last_consumption.columns = ['SK_MATRICULA', 'LAST_CONSUMPTION']

    # Merge mean and last consumption
    mean_consumption_2025 = pd.merge(mean_consumption, last_consumption, on='SK_MATRICULA', how='outer')

    # Calculate the MAX between mean and last consumption
    mean_consumption_2025['HCLQTCON'] = mean_consumption_2025[['MEAN_CONSUMPTION', 'LAST_CONSUMPTION']].max(axis=1)
    print('Consumption 2025:',mean_consumption_2025['HCLQTCON'].sum())
    # Rename column for clarity in GAMA/ABM simulation
    #mean_consumption_2025.rename(columns={'HCLQTCON': 'initial_mean_consumption'}, inplace=True)

    # ==========================================
    # 4. Saving Results
    # ==========================================
    mean_consumption_2025.to_csv(OUTPUT_CSV, index=False, sep=';')

    print(f"Success! Base year average saved to: {OUTPUT_CSV}")
    print(f"Total unique water meters processed: {len(mean_consumption_2025)}")
    
    
if __name__ == "__main__":
    main()

--- Calculating Base Year Consumption Average (2025) ---


Total records found for the year 2025: 186726
Consumption 2025: 208739.625974026


Success! Base year average saved to: ..\includes\Tabela_consumo_medio_Itapua_2025_12m.csv
Total unique water meters processed: 16559
